# CIFAR-10 Hypercontractive-Time Experiment

In [ ]:

import torch, math, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 128
EPOCHS = 10
TAU_MAX = 6.0


In [ ]:

class DiffusionSchedules:
    def __init__(self, tau_max=6.0):
        self.tau_max = tau_max

    def linear_tau(self, t):
        return self.tau_max * (t ** 2)

    def cosine_tau(self, t, s=0.008):
        f_t = torch.cos(((t + s) / (1 + s)) * (math.pi / 2)) ** 2
        tau_t = -torch.log(f_t.clamp(min=1e-5))
        tau_t = tau_t / tau_t.max() * self.tau_max
        return tau_t

    def front_loaded_tau(self, t, c=5.0):
        return self.tau_max * (1 - torch.exp(-c * t))


In [ ]:

class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        dim = 128
        self.net = nn.Sequential(
            nn.Conv2d(3, dim, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(dim, dim, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(dim, 3, 1)
        )
    def forward(self, x, t):
        return self.net(x)


In [ ]:

def train(model, loader, tau_fn):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    for _ in range(EPOCHS):
        for x,_ in loader:
            x = x.to(DEVICE)
            b = x.size(0)
            t = torch.rand(b, device=DEVICE)
            tau = tau_fn(t).view(-1,1,1,1)
            eps = torch.randn_like(x)
            x_t = torch.exp(-tau)*x + torch.sqrt(1 - torch.exp(-2*tau))*eps
            pred = model(x_t, t)
            loss = F.mse_loss(pred, eps)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


In [ ]:

def probe(model, loader, tau_fn):
    model.eval()
    t_bins = torch.linspace(0.02, 0.98, 30)
    L,G = [],[]
    for t_val in t_bins:
        tl,tg,n = 0,0,0
        for i,(x,_) in enumerate(loader):
            if i>=3: break
            x = x.to(DEVICE)
            b = x.size(0)
            t = torch.full((b,), t_val, device=DEVICE)
            tau = tau_fn(t).view(-1,1,1,1)
            eps = torch.randn_like(x)
            x_t = torch.exp(-tau)*x + torch.sqrt(1 - torch.exp(-2*tau))*eps
            pred = model(x_t,t)
            std = torch.sqrt(1 - torch.exp(-2*tau))
            loss = F.mse_loss(-pred/std, -eps/std)
            model.zero_grad(); loss.backward()
            grad = sum(p.grad.norm().item()**2 for p in model.parameters() if p.grad is not None)**0.5
            tl+=loss.item(); tg+=grad; n+=1
        L.append(tl/n); G.append(tg/n)
    return t_bins.numpy(), np.array(L), np.array(G)


In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:

sched = DiffusionSchedules(TAU_MAX)

m1 = train(SimpleUNet(), loader, sched.linear_tau)
m2 = train(SimpleUNet(), loader, sched.cosine_tau)
m3 = train(SimpleUNet(), loader, sched.front_loaded_tau)

t,l1,g1 = probe(m1, loader, sched.linear_tau)
_,l2,g2 = probe(m2, loader, sched.cosine_tau)
_,l3,g3 = probe(m3, loader, sched.front_loaded_tau)

tau1 = sched.linear_tau(torch.tensor(t)).numpy()
tau2 = sched.cosine_tau(torch.tensor(t)).numpy()
tau3 = sched.front_loaded_tau(torch.tensor(t)).numpy()

plt.figure(figsize=(10,6))
plt.subplot(121)
plt.plot(t,l1,label='linear'); plt.plot(t,l2,label='cos'); plt.plot(t,l3,label='front')
plt.title("Loss vs t"); plt.legend()

plt.subplot(122)
plt.plot(tau1,l1,label='linear'); plt.plot(tau2,l2,label='cos'); plt.plot(tau3,l3,label='front')
plt.title("Loss vs tau"); plt.legend()

plt.show()
